# Advanced lab 2 — Foundry A2A and least-data handoffs (preview)

Use Agent2Agent (A2A) for independently deployed agent boundaries. Prefer in-process composition or Foundry Workflows for tightly coupled steps. Delegate a scoped task, not an unrestricted conversation transcript.

Foundry currently supports A2A 1.0 and 0.3; new integrations should target 1.0. The v1.0 preview is JSON-RPC-only, text-only, nonstreaming, and Microsoft Entra authenticated. Sources: [incoming A2A](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/enable-agent-to-agent-endpoint), [outbound A2A tool](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/agent-to-agent), and the [A2A v1.0 specification](https://a2a-protocol.org/latest/specification/).

In [ ]:
import importlib.util
import json
import sys
from pathlib import Path
from urllib.parse import urlsplit

from pydantic import BaseModel, ConfigDict, Field

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

In [ ]:
class AgentSkillBoundary(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    id: str = Field(min_length=1, max_length=128)
    name: str = Field(min_length=1, max_length=128)
    description: str = Field(min_length=1, max_length=1000)


class AgentCardBoundary(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True, populate_by_name=True)

    name: str = Field(min_length=1, max_length=128)
    description: str = Field(min_length=1, max_length=2000)
    protocol_version: str = Field(alias="protocolVersion", pattern=r"^1\.0$")
    skills: tuple[AgentSkillBoundary, ...] = Field(min_length=1, max_length=50)


class HandoffEnvelope(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True)

    case_id: str = Field(min_length=1, max_length=128)
    task: str = Field(min_length=1, max_length=2000)
    approved_context_ids: tuple[str, ...] = Field(max_length=20)
    idempotency_key: str | None = Field(default=None, max_length=128)
    timeout_seconds: int = Field(default=30, ge=1, le=120)
    side_effect_allowed: bool = False


synthetic_card = AgentCardBoundary.model_validate(
    {
        "name": "policy-specialist",
        "description": "Answers questions about approved platform policy.",
        "protocolVersion": "1.0",
        "skills": [
            {
                "id": "policy-lookup",
                "name": "Policy lookup",
                "description": "Returns cited, read-only policy guidance.",
            }
        ],
    }
)
synthetic_card.model_dump(by_alias=True)

In [ ]:
project_host = urlsplit(session.project_endpoint).hostname
a2a_host = urlsplit(session.a2a_base_url).hostname
assert a2a_host == project_host
assert session.a2a_agent_card_url.endswith("/agentCard/v1.0")
{
    "a2a_ready": session.a2a_ready,
    "a2a_base_url": session.a2a_base_url,
    "agent_card_url": session.a2a_agent_card_url,
}

In [ ]:
a2a_cases = [
    json.loads(line)
    for line in (curriculum_root / "data" / "a2a_cases.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]


def route_request(request: str) -> str:
    lowered = request.lower()
    if any(word in lowered for word in ("delete", "unapproved", "pdf artifact")):
        return "local"
    if "policy" in lowered:
        return "policy-specialist"
    if "research" in lowered:
        return "research-specialist"
    if "ticket" in lowered:
        return "workflow-specialist"
    return "local"


routing_evidence = [
    {
        "case_id": case["case_id"],
        "expected": case["expectations"]["expected_route"],
        "actual": route_request(case["inputs"]["request"]),
    }
    for case in a2a_cases
]
assert all(item["expected"] == item["actual"] for item in routing_evidence)
routing_evidence

In [ ]:
handoff = HandoffEnvelope(
    case_id="a2a-minimum-context-01",
    task="Compare control A and control B using the approved policy excerpts.",
    approved_context_ids=("policy-control-a", "policy-control-b"),
    timeout_seconds=30,
    side_effect_allowed=False,
)
assert set(handoff.model_dump()) == {
    "case_id",
    "task",
    "approved_context_ids",
    "idempotency_key",
    "timeout_seconds",
    "side_effect_allowed",
}
handoff.model_dump()

In [ ]:
RUN_A2A_CONNECTED = False


async def invoke_pre_enabled_a2a() -> dict[str, object]:
    import httpx
    from a2a.client import A2ACardResolver, ClientConfig, create_client
    from a2a.helpers import new_text_message
    from a2a.types.a2a_pb2 import Role, SendMessageRequest
    from azure.identity import DefaultAzureCredential

    if not session.a2a_ready:
        raise RuntimeError("Configure foundry.a2a.remote_agent_name first.")
    credential = DefaultAzureCredential()
    try:
        token = credential.get_token("https://ai.azure.com/.default").token
        async with httpx.AsyncClient(
            headers={"Authorization": f"Bearer {token}"},
            timeout=httpx.Timeout(120.0),
        ) as http_client:
            resolver = A2ACardResolver(
                httpx_client=http_client,
                base_url=session.a2a_base_url,
                agent_card_path="agentCard/v1.0",
            )
            card = await resolver.get_agent_card()
            client = await create_client(
                agent=card,
                client_config=ClientConfig(streaming=False, httpx_client=http_client),
            )
            try:
                request = SendMessageRequest(
                    message=new_text_message(handoff.task, role=Role.ROLE_USER)
                )
                response_count = 0
                async for _response in client.send_message(request):
                    response_count += 1
                return {
                    "protocol_version": "1.0",
                    "skill_count": len(card.skills),
                    "response_events": response_count,
                }
            finally:
                await client.close()
    finally:
        credential.close()


if RUN_A2A_CONNECTED:
    a2a_result = await invoke_pre_enabled_a2a()
    print(a2a_result)
else:
    print("A2A call skipped; this lab never enables or provisions the endpoint.")

In [ ]:
RUN_FOUNDRY_A2A_TOOL = False

if RUN_FOUNDRY_A2A_TOOL:
    from azure.ai.projects import AIProjectClient
    from azure.ai.projects.models import A2APreviewTool
    from azure.identity import DefaultAzureCredential

    connection_name = session.labs.a2a.connection_name
    if connection_name.startswith("replace-"):
        raise RuntimeError("Configure the pre-provisioned A2A connection name.")
    with (
        DefaultAzureCredential() as credential,
        AIProjectClient(
            endpoint=session.project_endpoint, credential=credential
        ) as project,
    ):
        connection = project.connections.get(connection_name)
        a2a_tool = A2APreviewTool(project_connection_id=connection.id)
        print(
            {
                "tool_type": type(a2a_tool).__name__,
                "connection_name": connection_name,
            }
        )
else:
    print("Read-only A2A tool lookup skipped; provisioning stays external.")

## Exit criteria

Validate the authenticated v1.0 Agent Card, approved host, skill allow-list, timeout, retry budget, idempotency, and least-data envelope. Trace orchestrator → remote agent → orchestrator as an expected trajectory. Never treat a reachable Agent Card as trusted, and never delegate a side effect without local authorization and approval.